In [ ]:
# In[1]:

import pandas as pd

# Load metric.csv and parse timestamps as UTC
df_metric = pd.read_csv('metric.csv')
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Group by (cmdb_id, kpi_name)
group = df_metric.groupby(['cmdb_id', 'kpi_name'])

# Store full per-(cmdb_id, kpi_name) series in memory for later steps (do not print)
# Each value is a DataFrame with timestamp and value for that pair
series_store = { key: g[['timestamp', 'value']].copy() for key, g in group }

# Compute requested statistics per pair using the entire series (no filtering)
agg = group['value'].agg(
    count='count',
    mean='mean',
    min='min',
    max='max',
    p50=lambda x: x.quantile(0.5),
    p75=lambda x: x.quantile(0.75),
    p90=lambda x: x.quantile(0.90),
    p95=lambda x: x.quantile(0.95)
).reset_index()

# Sort for compact display (by count desc, then max desc) and limit to top 50 rows
result_df = agg.sort_values(by=['count', 'max'], ascending=[False, False]).head(50)

# Total number of unique (cmdb_id, kpi_name) pairs
total_pairs = agg.shape[0]

# Sample up to 5 distinct cmdb_id values
sample_cmdbs = df_metric['cmdb_id'].drop_duplicates().sample(n=min(5, df_metric['cmdb_id'].nunique()), random_state=1).reset_index(drop=True)

# Display compact aggregate table, total count, and sample cmdb_ids
result_df, total_pairs, sample_cmdbs

```
Out[1]:
```
```python
# Build a plain-English summary from previous results (reuse variables from earlier execution)
# Variables available: agg, result_df, total_pairs, sample_cmdbs

# Prepare top 3 memory KPI examples by mean
top_mem = agg[agg['kpi_name'].str.contains('mem', na=False)].sort_values(by='mean', ascending=False).head(3)
top_mem_lines = []
for _, r in top_mem.iterrows():
    top_mem_lines.append(f"{r['cmdb_id']} (kpi={r['kpi_name']}): mean={int(round(r['mean'])):,}, max={int(round(r['max'])):,}")

# Prepare a few example KPI types present (sample)
example_kpis = agg['kpi_name'].drop_duplicates().tolist()[:6]

summary = (
    f"Summary:\n"
    f"- Total unique (cmdb_id, kpi_name) pairs: {total_pairs}\n"
    f"- Sample cmdb_id values (up to 5): {', '.join(sample_cmdbs.tolist())}\n"
    f"- Top memory KPIs by mean (examples):\n  - " + "\n  - ".join(top_mem_lines) + "\n"
    f"- KPI types present (sample): {', '.join(example_kpis)}\n"
    f"- The compact aggregate table (top 50 rows) was produced as 'result_df'.\n"
    f"You can request more detail (e.g., full table, specific cmdb_id series, or more rows) if needed."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count          mean           min           max           p50           p75           p90           p95
53  productcatalogservice         mem     25  7.092356e+07  1.037866e+07  1.341297e+08  6.581815e+07  1.340137e+08  1.341187e+08  1.341293e+08
4               adservice         mem     25  1.128801e+08  1.127647e+08  1.131155e+08  1.128612e+08  1.129409e+08  1.129909e+08  1.130340e+08
60  recommendationservice         mem     25  4.539006e+07  4.527411e+07  4.557278e+07  4.536716e+07  4.540928e+07  4.549004e+07  4.552567e+07
10            cartservice         mem     25  4.364752e+07  4.358076e+07  4.376515e+07  4.365394e+07  4.366950e+07  4.369726e+07  4.370297e+07
30           emailservice         mem     25  4.348261e+07  4.346675e+07  4.351181e+07  4.348314e+07  4.349174e+07  4.349342e+07  4.349377e+07
..                    ...         ...    ...           ...           ...           ...           ...           ...           ...           ...
14        checkoutservice  latency-50     25  2.885610e-01  2.017886e-01  3.897234e-01  2.752980e-01  3.233933e-01  3.483048e-01  3.652876e-01
35               frontend  latency-50     25  2.610789e-01  1.733933e-01  3.678875e-01  2.105041e-01  3.425339e-01  3.464445e-01  3.544307e-01
26           emailservice         cpu     25  2.937553e-01  2.636146e-01  3.142583e-01  2.968416e-01  3.035463e-01  3.063812e-01  3.095632e-01
63                  redis         cpu     25  2.610083e-01  2.469614e-01  2.737605e-01  2.605476e-01  2.671867e-01  2.704673e-01  2.707805e-01
67        shippingservice         cpu     25  2.284500e-01  1.800747e-01  2.623397e-01  2.299407e-01  2.389291e-01  2.497806e-01  2.525495e-01

[50 rows x 10 columns], 73, 0    checkoutservice
1    currencyservice
2       emailservice
3              redis
4        cartservice
Name: cmdb_id, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Re-load metric.csv and parse timestamps as UTC (reuse variable name)
df_metric = pd.read_csv('metric.csv')
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Candidate components
candidates = [
    'checkoutservice',
    'currencyservice',
    'emailservice',
    'productcatalogservice',
    'recommendationservice'
]

# Restrict to candidate components
df_cand = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

# Global statistics per (cmdb_id, kpi_name) using full series (must be computed BEFORE any time-window filtering)
grp = df_cand.groupby(['cmdb_id', 'kpi_name'])
agg_global = grp['value'].agg(
    count='count',
    mean='mean',
    min='min',
    max='max',
    p50=lambda x: x.quantile(0.50),
    p75=lambda x: x.quantile(0.75),
    p90=lambda x: x.quantile(0.90),
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05)
).reset_index()

# Incident window (UTC)
start = pd.to_datetime('2024-01-15 23:03:19', utc=True)
end   = pd.to_datetime('2024-01-15 23:33:19', utc=True)

# Filter rows to the incident window
df_window = df_cand[(df_cand['timestamp'] >= start) & (df_cand['timestamp'] <= end)].copy()

# Window-level stats per pair
grp_w = df_window.groupby(['cmdb_id', 'kpi_name'])
agg_window = grp_w.agg(
    count_in_window=('value', 'count'),
    first_in_window=('timestamp', 'min'),
    last_in_window=('timestamp', 'max'),
    min_in_window=('value', 'min'),
    max_in_window=('value', 'max')
).reset_index()

# Merge global and window stats
merged = agg_global.merge(agg_window, on=['cmdb_id', 'kpi_name'], how='left')

# Fill missing count_in_window with 0; leave other window stats as NaN if no data in window
merged['count_in_window'] = merged['count_in_window'].fillna(0).astype(int)

# Order columns as requested
cols = [
    'cmdb_id', 'kpi_name', 'count', 'mean', 'min', 'max',
    'p50', 'p75', 'p90', 'p95', 'p5',
    'count_in_window', 'first_in_window', 'last_in_window', 'min_in_window', 'max_in_window'
]
final_df = merged[cols]

# Sort compactly for display (by count desc then cmdb_id)
final_df = final_df.sort_values(by=['count', 'cmdb_id'], ascending=[False, True]).head(50)

# Additional requested summaries
unique_pairs_count = agg_global.shape[0]
unique_kpis = pd.Series(df_cand['kpi_name'].drop_duplicates()).reset_index(drop=True).head(50)

# Display compact outputs: final table (top 50 rows), count of unique pairs, and up to 50 unique kpi names
final_df, unique_pairs_count, unique_kpis

```
Out[2]:
```
```python
# Build a concise plain-English summary from the previous results (reuse kernel variables)
# Available variables from prior step: final_df, unique_pairs_count, unique_kpis

import pandas as pd

# Compute summarized counts and time boundaries
df = final_df.copy()
total_pairs = int(unique_pairs_count)
kpi_list = unique_kpis.tolist()

n_with_any_window = int((df['count_in_window'] > 0).sum())
n_full_window = int((df['count_in_window'] == df['count']).sum())  # pairs with window count equal to global count
n_partial = int(((df['count_in_window'] > 0) & (df['count_in_window'] < df['count'])).sum())

# Earliest and latest timestamps observed within the incident window (among pairs that have window data)
firsts = pd.to_datetime(df['first_in_window'].dropna(), utc=True)
lasts  = pd.to_datetime(df['last_in_window'].dropna(), utc=True)
earliest_first = firsts.min().strftime('%Y-%m-%d %H:%M:%S %Z') if not firsts.empty else 'N/A'
latest_last    = lasts.max().strftime('%Y-%m-%d %H:%M:%S %Z') if not lasts.empty else 'N/A'

# Find pairs with partial coverage (count_in_window < count) and show up to 5 examples
partial_pairs = df[(df['count_in_window'] > 0) & (df['count_in_window'] < df['count'])][['cmdb_id','kpi_name','count','count_in_window']]
partial_examples = partial_pairs.head(5).to_dict(orient='records')

summary = (
    f"Summary for candidate components (checkoutservice, currencyservice, emailservice, "
    f"productcatalogservice, recommendationservice):\n\n"
    f"- Total unique (cmdb_id, kpi_name) pairs: {total_pairs}\n"
    f"- Unique KPI names present (up to 50): {', '.join(kpi_list)}\n\n"
    f"- Data in the incident window (2024-01-15 23:03:19 to 2024-01-15 23:33:19 UTC):\n"
    f"  • Pairs with any data in window: {n_with_any_window} of {total_pairs}\n"
    f"  • Pairs with full coverage in window (count_in_window == global count): {n_full_window}\n"
    f"  • Pairs with partial coverage in window: {n_partial}\n"
    f"  • Earliest timestamp observed in-window: {earliest_first}\n"
    f"  • Latest  timestamp observed in-window: {latest_last}\n\n"
    f"- Examples of pairs with partial coverage (up to 5): {partial_examples}\n\n"
    f"Notes:\n"
    f"- Most KPI series for these components have 25 samples and also show data inside the incident window (many first/last timestamps cluster around 2024-01-15 23:06:00 to 23:30:00 UTC).\n"
    f"- A few series have fewer samples (e.g., productcatalogservice diskio had fewer global samples and limited window coverage).\n"
    f"- For full per-pair numeric details, refer to the compact table 'final_df' produced earlier."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count          mean           min           max           p50           p75           p90           p95            p5  count_in_window           first_in_window            last_in_window  min_in_window  max_in_window
0         checkoutservice         cpu     25  3.850908e-01  2.764085e-01  4.912036e-01  3.825477e-01  4.246862e-01  4.573617e-01  4.807398e-01  2.931612e-01               25 2024-01-15 23:06:00+00:00 2024-01-15 23:30:00+00:00   2.764085e-01   4.912036e-01
1         checkoutservice  latency-50     25  2.885610e-01  2.017886e-01  3.897234e-01  2.752980e-01  3.233933e-01  3.483048e-01  3.652876e-01  2.375037e-01               25 2024-01-15 23:06:00+00:00 2024-01-15 23:30:00+00:00   2.017886e-01   3.897234e-01
2         checkoutservice  latency-90     25  7.633682e-01  5.503662e-01  9.597821e-01  7.607235e-01  8.411227e-01  9.124071e-01  9.341119e-01  5.680893e-01               25 2024-01-15 23:06:00+00:00 2024-01-15 23:30:00+00:00   5.503662e-01   9.597821e-01
3         checkoutservice         mem     25  1.200324e+07  1.183095e+07  1.218512e+07  1.199807e+07  1.209262e+07  1.214785e+07  1.217208e+07  1.184454e+07               25 2024-01-15 23:06:00+00:00 2024-01-15 23:30:00+00:00   1.183095e+07   1.218512e+07
4         checkoutservice      socket     25  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00               25 2024-01-15 23:06:00+00:00 2024-01-15 23:30:00+00:00   9.000000e+00   9.000000e+00
..                    ...         ...    ...           ...           ...           ...           ...           ...           ...           ...           ...              ...                       ...                       ...            ...            ...
31  recommendationservice  latency-90     25  2.786718e-02  9.545448e-03  6.224846e-02  1.274132e-02  4.822539e-02  5.650450e-02  5.761863e-02  9.559484e-03               25 2024-01-15 23:06:00+00:00 2024-01-15 23:30:00+00:00   9.545448e-03   6.224846e-02
32  recommendationservice         mem     25  4.539006e+07  4.527411e+07  4.557278e+07  4.536716e+07  4.540928e+07  4.549004e+07  4.552567e+07  4.531460e+07               25 2024-01-15 23:06:00+00:00 2024-01-15 23:30:00+00:00   4.527411e+07   4.557278e+07
33  recommendationservice      socket     25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00               25 2024-01-15 23:06:00+00:00 2024-01-15 23:30:00+00:00   4.000000e+00   4.000000e+00
34  recommendationservice    workload     25  1.763028e+01  1.643448e+01  1.897008e+01  1.739122e+01  1.822880e+01  1.854990e+01  1.879697e+01  1.662519e+01               25 2024-01-15 23:06:00+00:00 2024-01-15 23:30:00+00:00   1.643448e+01   1.897008e+01
21  productcatalogservice      diskio     13  8.157669e+09  3.574330e+09  9.079866e+09  8.440114e+09  8.782949e+09  8.849079e+09  8.942358e+09  6.182169e+09               13 2024-01-15 23:18:00+00:00 2024-01-15 23:30:00+00:00   3.574330e+09   9.079866e+09

[35 rows x 16 columns], 35, 0           cpu
1    latency-50
2    latency-90
3           mem
4        socket
5      workload
6         error
7        diskio
Name: kpi_name, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Reuse variables from previous steps: final_df, df_cand, df_window, df_metric
import pandas as pd
import numpy as np

# Ensure df_window exists and matches the incident window used previously
start = pd.to_datetime('2024-01-15 23:03:19', utc=True)
end   = pd.to_datetime('2024-01-15 23:33:19', utc=True)
if 'df_window' not in globals():
    df_window = df_cand[(df_cand['timestamp'] >= start) & (df_cand['timestamp'] <= end)].copy()
else:
    # recompute to be safe (idempotent)
    df_window = df_cand[(df_cand['timestamp'] >= start) & (df_cand['timestamp'] <= end)].copy()

# Prepare thresholds (global) from final_df: p95 and p5
thresholds = final_df[['cmdb_id', 'kpi_name', 'p95', 'p5']].copy()

# Merge window datapoints with thresholds so each row knows p95/p5
merged_win = df_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left', suffixes=('','_th'))

# Mark anomalies in-window relative to global thresholds
merged_win['above_p95'] = merged_win['value'] >= merged_win['p95']
merged_win['below_p5']  = merged_win['value'] <= merged_win['p5']

# Aggregate per (cmdb_id, kpi_name) within the window
agg_win = merged_win.groupby(['cmdb_id', 'kpi_name']).agg(
    count_in_window=('value', 'count'),
    count_above_p95=('above_p95', 'sum'),
    earliest_above_ts=('timestamp', lambda x: x[x.index[merged_win.loc[x.index, 'above_p95']].tolist()].min() if any(merged_win.loc[x.index, 'above_p95']) else pd.NaT),
    max_in_window=('value', 'max'),
    count_below_p5=('below_p5', 'sum'),
    earliest_below_ts=('timestamp', lambda x: x[x.index[merged_win.loc[x.index, 'below_p5']].tolist()].min() if any(merged_win.loc[x.index, 'below_p5']) else pd.NaT)
).reset_index()

# The lambda approach above is complex due to the grouped index mapping; use a clearer method:
# Recompute agg_win more straightforwardly:
if True:
    # compute counts
    grp = merged_win.groupby(['cmdb_id', 'kpi_name'])
    agg_counts = grp.agg(
        count_in_window=('value', 'count'),
        count_above_p95=('above_p95', 'sum'),
        max_in_window=('value', 'max'),
        count_below_p5=('below_p5', 'sum')
    ).reset_index()
    # earliest timestamps
    def first_timestamp_where(dfg, cond_col):
        sub = dfg[dfg[cond_col]]
        return sub['timestamp'].min() if not sub.empty else pd.NaT
    earliest_rows = []
    for (cmdb, kpi), dfg in merged_win.groupby(['cmdb_id','kpi_name']):
        ea = first_timestamp_where(dfg, 'above_p95')
        eb = first_timestamp_where(dfg, 'below_p5')
        earliest_rows.append({'cmdb_id': cmdb, 'kpi_name': kpi, 'earliest_above_ts': ea, 'earliest_below_ts': eb})
    df_earliest = pd.DataFrame(earliest_rows)
    agg_win = agg_counts.merge(df_earliest, on=['cmdb_id','kpi_name'], how='left')

# Now ensure we include pairs that had no rows in the window: left-merge with final_df pairs
pairs = final_df[['cmdb_id','kpi_name','p95','p5','count']].copy()
out = pairs.merge(agg_win, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with 0, preserve max_in_window as from merged data (may be NaN if no rows)
out['count_in_window'] = out['count_in_window'].fillna(0).astype(int)
out['count_above_p95'] = out['count_above_p95'].fillna(0).astype(int)
out['count_below_p5']  = out['count_below_p5'].fillna(0).astype(int)
out['max_in_window'] = out['max_in_window']  # keep NaN if no data
# earliest timestamps keep as datetime (NaT when missing)

# Convert earliest timestamps to ISO strings for display, keep NaT as NaN
out['earliest_above_p95'] = out['earliest_above_ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat() if pd.notna(x) else np.nan)
out['earliest_below_p5']  = out['earliest_below_ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat() if pd.notna(x) else np.nan)

# Prepare final compact table with requested columns
display_cols = [
    'cmdb_id', 'kpi_name',
    'count_in_window', 'count_above_p95', 'earliest_above_p95',
    'max_in_window', 'p95',
    'count_below_p5', 'earliest_below_p5', 'p5'
]
out_final = out[display_cols].copy()

# Sort by total anomalies descending
out_final['total_anomalies'] = out_final['count_above_p95'] + out_final['count_below_p5']
out_final = out_final.sort_values(by=['total_anomalies', 'count_above_p95'], ascending=[False, False]).head(20)

# Compute summary metrics requested
total_pairs_with_anomaly = int(((out['count_above_p95'] + out['count_below_p5']) > 0).sum())

# Earliest anomaly timestamp across all pairs (consider both above and below); collect datetimes
all_earliest_ts = pd.concat([
    out['earliest_above_ts'].dropna(),
    out['earliest_below_ts'].dropna()
])
earliest_anomaly_ts = all_earliest_ts.min() if not all_earliest_ts.empty else pd.NaT
earliest_anomaly_iso = earliest_anomaly_ts.isoformat() if pd.notna(earliest_anomaly_ts) else np.nan

# Keep output compact: top20 table, total count, earliest timestamp
out_final = out_final.drop(columns=['total_anomalies'])  # remove helper column for display
out_final, total_pairs_with_anomaly, earliest_anomaly_iso

```
Out[3]:
```
Summary of incident-window anomaly analysis (using global p95/p5 thresholds):

- Scope: candidate components = checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice. Total pairs analyzed = 35.
- Number of pairs with any anomaly (value >= p95 or <= p5) in the incident window: 35 (all pairs).
- Earliest anomaly observed: 2024-01-15T23:06:00+00:00 (UTC).

Top offending (cmdb_id, kpi_name) pairs by anomaly counts (count_above_p95, count_below_p5) — top 10 examples:
1. checkoutservice — socket: above_p95=25, below_p5=25 (earliest above: 2024-01-15T23:06:00Z)
2. currencyservice — error: above_p95=25, below_p5=25 (earliest above: 2024-01-15T23:06:00Z)
3. currencyservice — socket: above_p95=25, below_p5=25 (earliest above: 2024-01-15T23:06:00Z)
4. emailservice — diskio: above_p95=25, below_p5=25 (earliest above: 2024-01-15T23:06:00Z)
5. productcatalogservice — error: above_p95=25, below_p5=25 (earliest above: 2024-01-15T23:06:00Z)
6. recommendationservice — diskio: above_p95=25, below_p5=25 (earliest above: 2024-01-15T23:06:00Z)
7. recommendationservice — socket: above_p95=25, below_p5=25 (earliest above: 2024-01-15T23:06:00Z)
8. emailservice — socket: above_p95=25, below_p5=24 (earliest above: 2024-01-15T23:06:00Z)
9. productcatalogservice — socket: above_p95=12, below_p5=12 (earliest above: 2024-01-15T23:19:00Z)
10. emailservice — latency-50: above_p95=2, below_p5=12 (earliest above: 2024-01-15T23:22:00Z)

Notes/interpretation:
- Many series are fully flagged because their values equal the global p95 and/or p5 (discrete metrics like socket/error/diskio), which causes both above_p95 and below_p5 counts to be nonzero. Treat those carefully — they may reflect constant or categorical metrics rather than directional degradations.
- Time clustering: the earliest anomalous points for many series are at 2024-01-15T23:06:00Z (the start of anomaly activity in the window).
- Numeric KPIs with directional anomalies include CPU, latency, mem, workload for checkoutservice and currencyservice (a few samples exceed p95 or fall below p5).

If you want, next steps I can run:
- Show the full top-20 anomaly table or expand to include exact counts and timestamps per pair.
- Drill into specific pairs (e.g., checkoutservice CPU/latency/mem) and list the exact anomalous datapoints.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count_in_window  count_above_p95         earliest_above_p95  max_in_window           p95  count_below_p5          earliest_below_p5            p5
4         checkoutservice      socket               25               25  2024-01-15T23:06:00+00:00   9.000000e+00  9.000000e+00              25  2024-01-15T23:06:00+00:00  9.000000e+00
7         currencyservice       error               25               25  2024-01-15T23:06:00+00:00   0.000000e+00  0.000000e+00              25  2024-01-15T23:06:00+00:00  0.000000e+00
11        currencyservice      socket               25               25  2024-01-15T23:06:00+00:00   4.000000e+00  4.000000e+00              25  2024-01-15T23:06:00+00:00  4.000000e+00
14           emailservice      diskio               25               25  2024-01-15T23:06:00+00:00   0.000000e+00  0.000000e+00              25  2024-01-15T23:06:00+00:00  0.000000e+00
21  productcatalogservice       error               25               25  2024-01-15T23:06:00+00:00   0.000000e+00  0.000000e+00              25  2024-01-15T23:06:00+00:00  0.000000e+00
28  recommendationservice      diskio               25               25  2024-01-15T23:06:00+00:00   0.000000e+00  0.000000e+00              25  2024-01-15T23:06:00+00:00  0.000000e+00
32  recommendationservice      socket               25               25  2024-01-15T23:06:00+00:00   4.000000e+00  4.000000e+00              25  2024-01-15T23:06:00+00:00  4.000000e+00
18           emailservice      socket               25               25  2024-01-15T23:06:00+00:00   3.500000e+00  3.000000e+00              24  2024-01-15T23:06:00+00:00  3.000000e+00
25  productcatalogservice      socket               25               12  2024-01-15T23:19:00+00:00   7.000000e+00  7.000000e+00              12  2024-01-15T23:06:00+00:00  4.000000e+00
15           emailservice  latency-50               25                2  2024-01-15T23:22:00+00:00   3.146372e-03  3.102294e-03              12  2024-01-15T23:06:00+00:00  3.000000e-03
16           emailservice  latency-90               25                2  2024-01-15T23:22:00+00:00   4.863469e-03  4.784129e-03              12  2024-01-15T23:06:00+00:00  4.600000e-03
0         checkoutservice         cpu               25                2  2024-01-15T23:21:00+00:00   4.912036e-01  4.807398e-01               2  2024-01-15T23:17:00+00:00  2.931612e-01
1         checkoutservice  latency-50               25                2  2024-01-15T23:23:00+00:00   3.897234e-01  3.652876e-01               2  2024-01-15T23:06:00+00:00  2.375037e-01
2         checkoutservice  latency-90               25                2  2024-01-15T23:19:00+00:00   9.597821e-01  9.341119e-01               2  2024-01-15T23:09:00+00:00  5.680893e-01
3         checkoutservice         mem               25                2  2024-01-15T23:09:00+00:00   1.218512e+07  1.217208e+07               2  2024-01-15T23:26:00+00:00  1.184454e+07
5         checkoutservice    workload               25                2  2024-01-15T23:21:00+00:00   1.102067e+00  1.098057e+00               2  2024-01-15T23:28:00+00:00  5.876433e-01
6         currencyservice         cpu               25                2  2024-01-15T23:18:00+00:00   1.921195e+01  1.899925e+01               2  2024-01-15T23:14:00+00:00  1.795832e+01
8         currencyservice  latency-50               25                2  2024-01-15T23:06:00+00:00   6.715215e-02  6.496236e-02               2  2024-01-15T23:14:00+00:00  5.854079e-02
9         currencyservice  latency-90               25                2  2024-01-15T23:06:00+00:00   1.287678e-01  1.160516e-01               2  2024-01-15T23:16:00+00:00  9.523756e-02
10        currencyservice         mem               25                2  2024-01-15T23:23:00+00:00   4.119757e+07  4.108102e+07               2  2024-01-15T23:13:00+00:00  4.027980e+07, 35, '2024-01-15T23:06:00+00:00')```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse candidate components and incident window if present; otherwise define
try:
    candidates
except NameError:
    candidates = [
        'checkoutservice',
        'currencyservice',
        'emailservice',
        'productcatalogservice',
        'recommendationservice'
    ]
try:
    start, end
except NameError:
    start = pd.to_datetime('2024-01-15 23:03:19', utc=True)
    end   = pd.to_datetime('2024-01-15 23:33:19', utc=True)

# Load trace.csv and parse timestamps as UTC
df_trace = pd.read_csv('trace.csv')
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Restrict to candidate components
df_trace_cand = df_trace[df_trace['cmdb_id'].isin(candidates)].copy()

# Compute global thresholds per (cmdb_id, trace_name) using full series (count, p95, p5)
grp_full = df_trace_cand.groupby(['cmdb_id', 'trace_name'])['value']
trace_global = grp_full.agg(
    count='count',
    p95=lambda x: x.quantile(0.95),
    p5 =lambda x: x.quantile(0.05)
).reset_index()

# Filter rows to incident window (keep for aggregation)
df_trace_win = df_trace_cand[(df_trace_cand['timestamp'] >= start) & (df_trace_cand['timestamp'] <= end)].copy()

# Merge window rows with global thresholds so each window row knows p95/p5
df_win_thresh = df_trace_win.merge(trace_global[['cmdb_id','trace_name','p95','p5']], on=['cmdb_id','trace_name'], how='left')

# Mark anomalies relative to global thresholds
df_win_thresh['above_p95'] = df_win_thresh['value'] >= df_win_thresh['p95']
df_win_thresh['below_p5']  = df_win_thresh['value'] <= df_win_thresh['p5']

# Aggregate per (cmdb_id, trace_name) within the window
grp_win = df_win_thresh.groupby(['cmdb_id','trace_name'])
agg_win = grp_win.agg(
    count_in_window = ('value','count'),
    count_above_p95 = ('above_p95','sum'),
    max_in_window   = ('value','max'),
    count_below_p5  = ('below_p5','sum')
).reset_index()

# Compute earliest timestamps for above/below conditions per pair
earliest_rows = []
for (cmdb, tname), dfg in df_win_thresh.groupby(['cmdb_id','trace_name']):
    ea = dfg.loc[dfg['above_p95'], 'timestamp'].min() if dfg['above_p95'].any() else pd.NaT
    eb = dfg.loc[dfg['below_p5'],  'timestamp'].min() if dfg['below_p5'].any() else pd.NaT
    earliest_rows.append({'cmdb_id': cmdb, 'trace_name': tname, 'earliest_above_ts': ea, 'earliest_below_ts': eb})
df_earliest = pd.DataFrame(earliest_rows)

# Merge agg_win with earliest timestamps
agg_win = agg_win.merge(df_earliest, on=['cmdb_id','trace_name'], how='left')

# Ensure all global pairs are present in output (pairs without window rows will be filled)
pairs = trace_global[['cmdb_id','trace_name','p95','p5','count']].copy()
trace_out = pairs.merge(agg_win, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for counts with 0 and keep NaT for earliest timestamps and NaN for max_in_window
trace_out['count_in_window'] = trace_out['count_in_window'].fillna(0).astype(int)
trace_out['count_above_p95'] = trace_out['count_above_p95'].fillna(0).astype(int)
trace_out['count_below_p5']  = trace_out['count_below_p5'].fillna(0).astype(int)

# Convert earliest timestamps to ISO strings for display
trace_out['earliest_above_p95'] = trace_out['earliest_above_ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat() if pd.notna(x) else np.nan)
trace_out['earliest_below_p5']  = trace_out['earliest_below_ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat() if pd.notna(x) else np.nan)

# Prepare final compact table with requested columns
trace_out_final = trace_out[[
    'cmdb_id','trace_name',
    'count_in_window','count_above_p95','earliest_above_p95',
    'max_in_window','p95',
    'count_below_p5','earliest_below_p5','p5'
]].copy()

# Sort by count_above_p95 descending and limit to top 20
trace_out_final = trace_out_final.sort_values(by=['count_above_p95','count_below_p5'], ascending=[False, False]).head(20)

# Summary metrics: number of pairs with any anomaly and earliest anomaly timestamp across all trace pairs
trace_out['total_anomalies'] = trace_out['count_above_p95'] + trace_out['count_below_p5']
total_trace_pairs_with_anomaly = int((trace_out['total_anomalies'] > 0).sum())

# Earliest anomaly across all pairs (consider both above and below earliest timestamps)
all_earliest_ts = pd.concat([
    trace_out['earliest_above_ts'].dropna(),
    trace_out['earliest_below_ts'].dropna()
])
earliest_trace_anomaly_ts = all_earliest_ts.min() if not all_earliest_ts.empty else pd.NaT
earliest_trace_anomaly_iso = earliest_trace_anomaly_ts.isoformat() if pd.notna(earliest_trace_anomaly_ts) else np.nan

# Return compact results
trace_out_final, total_trace_pairs_with_anomaly, earliest_trace_anomaly_iso

```
Out[4]:
```
Summary of trace analysis for candidate components (checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice):

- Total (cmdb_id, trace_name) pairs analyzed: 88.
- Number of pairs with any anomaly in the incident window: 88 (all pairs).
- Earliest anomaly observed: 2024-01-15T23:06:00+00:00 (UTC).

Key findings:
- The top entries are various trace error_rate metrics (e.g., trace.from_checkoutservice.error_rate, trace.from_frontendservice.error_rate, trace.from_root.error_rate, trace.to_* .error_rate) for the candidate services.
- Every trace error_rate series in the window was flagged: count_in_window = 25 and both count_above_p95 and count_below_p5 = 25 for many series.
- This pattern occurs because the global p95 and p5 thresholds for these series are 0.0, and the window values equal 0.0 — the threshold logic (value >= p95 and value <= p5) therefore marks points as both "above p95" and "below p5". This is a discrete/thresholding artifact rather than a clear directional degradation.

Interpretation and next steps:
- The trace error_rate metrics are uniformly flagged due to threshold equality at 0.0. Treat these anomaly counts cautiously.
- To get actionable insight, inspect raw trace values for nonzero error rates, check whether error_rate is truly nonzero at any time, and verify trace instrumentation/threshold computation.
- If you want, I can:
  - show the full trace table of top pairs,
  - list any trace series with nonzero values in-window, or
  - recompute anomalies using a stricter rule (e.g., value > p95 or value < p5) to avoid the equality artifact.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                   trace_name  count_in_window  count_above_p95         earliest_above_p95  max_in_window  p95  count_below_p5          earliest_below_p5   p5
2         checkoutservice        trace.from_checkoutservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
6         checkoutservice        trace.from_frontendservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
10        checkoutservice                   trace.from_root.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
14        checkoutservice          trace.to_checkoutservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
18        checkoutservice          trace.to_currencyservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
22        checkoutservice             trace.to_emailservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
26        checkoutservice           trace.to_paymentservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
30        checkoutservice    trace.to_productcatalogservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
34        currencyservice        trace.from_checkoutservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
38        currencyservice        trace.from_frontendservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
42        currencyservice                   trace.from_root.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
46           emailservice        trace.from_checkoutservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
50           emailservice                   trace.from_root.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
54  productcatalogservice        trace.from_checkoutservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
58  productcatalogservice        trace.from_frontendservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
62  productcatalogservice  trace.from_recommendationservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
66  productcatalogservice                   trace.from_root.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
70  recommendationservice        trace.from_frontendservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
74  recommendationservice  trace.from_recommendationservice.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0
78  recommendationservice                   trace.from_root.error_rate               25               25  2024-01-15T23:06:00+00:00            0.0  0.0              25  2024-01-15T23:06:00+00:00  0.0, 88, '2024-01-15T23:06:00+00:00')```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse incident window if present, otherwise define it
try:
    start, end
except NameError:
    start = pd.to_datetime('2024-01-15 23:03:19', utc=True)
    end   = pd.to_datetime('2024-01-15 23:33:19', utc=True)

# Load log.csv and parse timestamps as UTC
df_log = pd.read_csv('log.csv')
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Filter log rows to the incident window
df_log_window = df_log[(df_log['timestamp'] >= start) & (df_log['timestamp'] <= end)].copy()

# Group by cmdb_id and log_name and compute count, sum(value), mean(value)
log_group = (
    df_log_window
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(count='count', sum='sum', mean='mean')
    .reset_index()
    .sort_values(by='count', ascending=False)
)

# Top 20 rows by count (compact)
log_group_top20 = log_group.head(20)

# Unique cmdb_id values present in-window (compact)
unique_cmdbs_in_log_window = df_log_window['cmdb_id'].drop_duplicates().reset_index(drop=True)

# Load error_logs.csv and parse timestamps as UTC
df_err = pd.read_csv('error_logs.csv')
df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True)

# Filter error logs to the incident window
df_err_window = df_err[(df_err['timestamp'] >= start) & (df_err['timestamp'] <= end)].copy()

# Group by cmdb_id: count and earliest timestamp in-window
err_group = (
    df_err_window
    .groupby('cmdb_id', dropna=False)
    .agg(count=('message', 'count'), earliest_in_window=('timestamp', 'min'))
    .reset_index()
)
# Format earliest timestamp to ISO string (UTC) for compact display
err_group['earliest_in_window'] = err_group['earliest_in_window'].dt.tz_convert('UTC').apply(lambda x: x.isoformat() if pd.notna(x) else pd.NaT)

# Sample up to 20 error messages (include timestamp and cmdb_id), sorted by timestamp
sample_messages = (
    df_err_window.sort_values('timestamp')[['timestamp', 'cmdb_id', 'message']].head(20).copy()
)
sample_messages['timestamp'] = sample_messages['timestamp'].dt.tz_convert('UTC').apply(lambda x: x.isoformat())

# Return compact outputs
log_group_top20, unique_cmdbs_in_log_window, err_group, sample_messages

```
Out[5]:
```
Summary of logs in the incident window (2024-01-15 23:03:19 — 2024-01-15 23:33:19 UTC):

- Which services had log activity in-window (unique cmdb_id):  
  adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, shippingservice, redis.

- log.csv overview (grouped by cmdb_id, log_name):  
  - Every service shows 25 samples for log.row_count and log.error_count in most services (redis has 5 samples).  
  - log.error_count is 0 for all listed services (no aggregated errors in the normalized log metric).  
  - Notable log.row_count totals (sum over the window):  
    - frontend: sum = 60,130 (mean 2,405.20)  
    - currencyservice: sum = 53,475 (mean 2,139.00)  
    - cartservice: sum = 17,402 (mean 696.08)  
    - recommendationservice: sum = 12,696 (mean 507.84)  
    - adservice: sum = 10,173 (mean 406.92)  
    - shippingservice: sum = 10,084 (mean 403.36)  
    - checkoutservice: sum = 1,900 (mean 76.00)  
    - paymentservice: sum = 1,266 (mean 50.64)  
    - emailservice: sum = 633 (mean 25.32)  
    - redis: sum = 25 over 5 samples (mean 5.00)

- error_logs.csv in-window: no error log records were found in the incident window (no grouped counts, no sample messages).

Interpretation / quick takeaways:
- There are high request/log volumes for frontend and currencyservice during the window; however, normalized log.error_count is zero across services, and there are no raw error log entries in error_logs.csv for the window.  
- This suggests no obvious log-reported errors during the incident window, though high traffic on frontend/currencyservice could correlate with other telemetry anomalies seen earlier (metrics/traces).  
- If you want, next steps: inspect raw log lines (if available outside the normalized counts), or correlate these high log volumes with the metric/trace anomalies already found.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id         log_name  count    sum     mean
0               adservice  log.error_count     25      0     0.00
1               adservice    log.row_count     25  10173   406.92
2             cartservice  log.error_count     25      0     0.00
3             cartservice    log.row_count     25  17402   696.08
4         checkoutservice  log.error_count     25      0     0.00
5         checkoutservice    log.row_count     25   1900    76.00
6         currencyservice  log.error_count     25      0     0.00
7         currencyservice    log.row_count     25  53475  2139.00
8            emailservice  log.error_count     25      0     0.00
9            emailservice    log.row_count     25    633    25.32
10               frontend  log.error_count     25      0     0.00
11               frontend    log.row_count     25  60130  2405.20
12         paymentservice  log.error_count     25      0     0.00
13         paymentservice    log.row_count     25   1266    50.64
14  recommendationservice  log.error_count     25      0     0.00
15  recommendationservice    log.row_count     25  12696   507.84
19        shippingservice    log.row_count     25  10084   403.36
18        shippingservice  log.error_count     25      0     0.00
17                  redis    log.row_count      5     25     5.00
16                  redis  log.error_count      5      0     0.00, 0                adservice
1              cartservice
2          checkoutservice
3          currencyservice
4             emailservice
5                 frontend
6           paymentservice
7    recommendationservice
8          shippingservice
9                    redis
Name: cmdb_id, dtype: object, Empty DataFrame
Columns: [cmdb_id, count, earliest_in_window]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse or load metric.csv and parse timestamps as UTC
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Candidate components and KPI names (numeric)
candidates = [
    'checkoutservice',
    'currencyservice',
    'emailservice',
    'productcatalogservice',
    'recommendationservice'
]
kpis_numeric = ['cpu', 'latency-50', 'latency-90', 'mem', 'workload']

# Filter to candidates and numeric KPIs
df_sel = df_metric[
    df_metric['cmdb_id'].isin(candidates) &
    df_metric['kpi_name'].isin(kpis_numeric)
].copy()

# Compute global thresholds (p95 and p5) PER pair using the full series (before window filtering)
thresholds = (
    df_sel
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
)

# Incident window (UTC)
start = pd.to_datetime('2024-01-15 23:03:19', utc=True)
end   = pd.to_datetime('2024-01-15 23:33:19', utc=True)

# Filter points to the incident window
df_window_sel = df_sel[(df_sel['timestamp'] >= start) & (df_sel['timestamp'] <= end)].copy()

# Compute per-pair extremes in-window
extremes = (
    df_window_sel
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(max_in_window=('value','max'), min_in_window=('value','min'))
)

# Merge thresholds with window extremes, include pairs that might have no points in-window
pairs = thresholds.merge(extremes, on=['cmdb_id','kpi_name'], how='left')

# For earliest timestamps of the extreme values, we will compute per pair
# Prepare a lookup grouped by pair for window rows
grouped_window = {k: g for k, g in df_window_sel.groupby(['cmdb_id','kpi_name'])}

rows = []
for _, r in pairs.iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    p95 = r['p95']
    p5  = r['p5']
    max_win = r['max_in_window']
    min_win = r['min_in_window']

    # default values
    severity = 0.0
    side_used = np.nan
    threshold_value = np.nan
    extreme_value_in_window = np.nan
    earliest_ts_iso = np.nan

    key = (cmdb, kpi)
    dfg = grouped_window.get(key, pd.DataFrame(columns=df_window_sel.columns))

    # compute above-side severity if applicable
    above_sev = -np.inf
    if pd.notna(max_win) and p95 is not None and p95 > 0:
        if max_win > p95:
            above_sev = (max_win - p95) / p95
        else:
            above_sev = 0.0
    else:
        above_sev = -np.inf  # mark invalid if threshold zero or missing

    # compute below-side severity if applicable
    below_sev = -np.inf
    if pd.notna(min_win) and p5 is not None and p5 > 0:
        if min_win < p5:
            below_sev = (p5 - min_win) / p5
        else:
            below_sev = 0.0
    else:
        below_sev = -np.inf

    # determine which side gives max severity
    # If both invalid (-inf) or both zero, severity is 0
    if above_sev == -np.inf and below_sev == -np.inf:
        severity = 0.0
    else:
        # Replace -inf with 0 for comparison if the other side is valid
        a = above_sev if above_sev != -np.inf else 0.0
        b = below_sev if below_sev != -np.inf else 0.0
        if a >= b and a > 0:
            severity = float(a)
            side_used = 'above'
            threshold_value = p95
            extreme_value_in_window = max_win
            # earliest timestamp when extreme occurred (value equals max_in_window)
            if not dfg.empty and pd.notna(max_win):
                sub = dfg[dfg['value'] == max_win]
                if not sub.empty:
                    earliest_ts_iso = sub['timestamp'].min().isoformat()
        elif b > 0:
            severity = float(b)
            side_used = 'below'
            threshold_value = p5
            extreme_value_in_window = min_win
            if not dfg.empty and pd.notna(min_win):
                sub = dfg[dfg['value'] == min_win]
                if not sub.empty:
                    earliest_ts_iso = sub['timestamp'].min().isoformat()
        else:
            severity = 0.0

    rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'severity': severity,
        'side_used': side_used,
        'threshold_value': threshold_value,
        'extreme_value_in_window': extreme_value_in_window,
        'earliest_timestamp_of_extreme': earliest_ts_iso
    })

results_df = pd.DataFrame(rows)

# Sort by severity desc and limit to top 10 rows for compact output
results_top10 = results_df.sort_values(by='severity', ascending=False).head(10)

# Count pairs with severity > 0.5
count_sev_over_50pct = int((results_df['severity'] > 0.5).sum())

# Display compact outputs
results_top10, count_sev_over_50pct

```
Out[6]:
```
```python
# Build a concise plain-English summary from previous results (reuse variables: results_top10, count_sev_over_50pct)
# Display as a single string variable per task rules.

top = results_top10.reset_index(drop=True)

lines = []
lines.append("Incident severity summary for selected numeric KPIs (candidate services):")
lines.append(f"- Number of (cmdb_id, kpi_name) pairs with severity > 50%: {count_sev_over_50pct}")
lines.append("- Top severity cases (severity = relative breach; 'below' means value dropped below p5, 'above' means exceeded p95):")

for i, row in top.iterrows():
    sev_pct = f"{row['severity']*100:.1f}%"
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    side = row['side_used'] if pd.notna(row['side_used']) else 'none'
    thresh = row['threshold_value']
    extreme = row['extreme_value_in_window']
    ts = row['earliest_timestamp_of_extreme'] if pd.notna(row['earliest_timestamp_of_extreme']) else 'N/A'
    lines.append(f"  {i+1}. {cmdb} - {kpi}: severity={sev_pct}, side={side}, threshold={thresh}, extreme={extreme}, time={ts}")

lines.append("")
lines.append("Key takeaways:")
lines.append("- No pair exceeded a 50% severity breach; the largest observed severity is ~19.2% (emailservice workload, below p5).")
lines.append("- Most notable deviations are 'below' the p5 (drops) for emailservice and checkoutservice workload/latency/cpu, and smaller 'above' breaches for some latency and workload series in currencyservice/recommendationservice.")
lines.append("- Overall severities are modest (<0.2), suggesting limited relative deviation from global thresholds during the incident window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  severity side_used  threshold_value  extreme_value_in_window earliest_timestamp_of_extreme
14           emailservice    workload  0.191609     below         0.573423                 0.463550     2024-01-15T23:30:00+00:00
1         checkoutservice  latency-50  0.150377     below         0.237504                 0.201789     2024-01-15T23:16:00+00:00
7         currencyservice  latency-90  0.109573     above         0.116052                 0.128768     2024-01-15T23:21:00+00:00
4         checkoutservice    workload  0.092647     below         0.587643                 0.533200     2024-01-15T23:30:00+00:00
22  recommendationservice  latency-90  0.080353     above         0.057619                 0.062248     2024-01-15T23:30:00+00:00
0         checkoutservice         cpu  0.057145     below         0.293161                 0.276408     2024-01-15T23:30:00+00:00
15  productcatalogservice         cpu  0.043822     below         3.143884                 3.006113     2024-01-15T23:06:00+00:00
9         currencyservice    workload  0.035059     above        77.299223                80.009267     2024-01-15T23:21:00+00:00
20  recommendationservice         cpu  0.034197     below         2.256029                 2.178880     2024-01-15T23:20:00+00:00
6         currencyservice  latency-50  0.033708     above         0.064962                 0.067152     2024-01-15T23:21:00+00:00, 0)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse existing variables if present, otherwise define/load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure timestamps are parsed (idempotent)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Candidate components and incident window (reuse if available)
try:
    candidates
except NameError:
    candidates = [
        'checkoutservice',
        'currencyservice',
        'emailservice',
        'productcatalogservice',
        'recommendationservice'
    ]
try:
    start, end
except NameError:
    start = pd.to_datetime('2024-01-15 23:03:19', utc=True)
    end   = pd.to_datetime('2024-01-15 23:33:19', utc=True)

# Select trace rows for candidate components and trace_name containing 'duration' or 'latency' (case-insensitive)
mask_comp = df_trace['cmdb_id'].isin(candidates)
mask_name = df_trace['trace_name'].str.contains('duration|latency', case=False, na=False)
df_trace_sel = df_trace[mask_comp & mask_name].copy()

# Compute global p95 per (cmdb_id, trace_name) using full series (must be BEFORE window filtering)
trace_global = (
    df_trace_sel
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count='count', p95=lambda x: x.quantile(0.95))
)

# Filter trace points to the incident window
df_trace_win = df_trace_sel[(df_trace_sel['timestamp'] >= start) & (df_trace_sel['timestamp'] <= end)].copy()

# Merge window points with global p95 so each window row knows its p95
df_win_with_p95 = df_trace_win.merge(trace_global[['cmdb_id','trace_name','p95']], on=['cmdb_id','trace_name'], how='left')

# Mark points strictly above p95
df_win_with_p95['above_p95'] = df_win_with_p95['value'] > df_win_with_p95['p95']

# Aggregate per pair within the window
agg_win = df_win_with_p95.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    count_in_window=('value','count'),
    count_above_p95=('above_p95','sum'),
    max_in_window=('value','max')
)

# Earliest timestamp where value > p95 per pair
earliest_above = (
    df_win_with_p95[df_win_with_p95['above_p95']]
    .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_above_ts'})
)

# Combine aggregates with earliest timestamps and ensure all global pairs are present
trace_pairs = trace_global[['cmdb_id','trace_name','p95']].copy()
trace_out = trace_pairs.merge(agg_win, on=['cmdb_id','trace_name'], how='left')
trace_out = trace_out.merge(earliest_above, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for counts with 0
trace_out['count_in_window'] = trace_out['count_in_window'].fillna(0).astype(int)
trace_out['count_above_p95'] = trace_out['count_above_p95'].fillna(0).astype(int)
# max_in_window may be NaN if no points in window; keep as NaN

# Convert earliest_above_ts to ISO strings (UTC) or NaN
trace_out['earliest_above_p95'] = trace_out['earliest_above_ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat() if pd.notna(x) else np.nan)

# Prepare final compact table with requested columns
trace_out_final = trace_out[[
    'cmdb_id','trace_name',
    'count_in_window','count_above_p95','earliest_above_p95',
    'max_in_window','p95'
]].copy()

# Sort by count_above_p95 descending and limit to top 20
trace_out_final = trace_out_final.sort_values(by=['count_above_p95','count_in_window'], ascending=[False, False]).head(20)

# Summary metrics: number of pairs with count_above_p95 > 0 and earliest among earliest_above_p95
total_pairs_with_anomaly = int((trace_out['count_above_p95'] > 0).sum())

# Find earliest non-null earliest_above_ts across all pairs
all_earliest = trace_out['earliest_above_ts'].dropna()
earliest_anomaly_ts = all_earliest.min() if not all_earliest.empty else pd.NaT
earliest_anomaly_iso = earliest_anomaly_ts.isoformat() if pd.notna(earliest_anomaly_ts) else np.nan

# Return compact results
trace_out_final, total_pairs_with_anomaly, earliest_anomaly_iso

```
Out[7]:
```
Summary of trace-duration/latency analysis for the candidate services (incident window 2024-01-15 23:03:19 — 23:33:19 UTC):

- Number of (cmdb_id, trace_name) pairs with any point above the global p95 in-window: 44.
- Earliest observed above-p95 timestamp across all trace pairs: 2024-01-15T23:07:00+00:00 (UTC).
- Pattern observed:
  - Many duration/latency trace series (especially for checkoutservice, and several for currencyservice) show a small number of points above their global p95 during the window. In the top rows, typical entries have count_in_window = 25 and count_above_p95 = 2 (i.e., two samples exceeded p95).
  - Affected trace names include variants like trace.from_checkoutservice.duration_mean, trace.from_frontendservice.duration_p95, trace.from_root.duration_mean/p95, and several trace.to_* .duration_* metrics on checkoutservice and currencyservice.
- Interpretation:
  - The issue is widespread across many trace duration metrics but generally modest in magnitude per series (only a couple of samples exceed p95 for most pairs).
  - The anomalous events started early in the window (first above-p95 around 23:07 UTC) and appear in multiple inter-service trace paths involving checkoutservice and currencyservice.
- Suggested next steps:
  - Inspect the specific trace samples around the earliest above-p95 timestamps (e.g., 23:07 UTC and the ~23:20–23:24 UTC spikes) to see which requests/spans were slow and whether they correlate with errors or increased workload.
  - Correlate these trace spikes with the metric anomalies (latency, CPU, workload) and log volumes already observed to identify likely root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                    trace_name  count_in_window  count_above_p95         earliest_above_p95  max_in_window       p95
0   checkoutservice      trace.from_checkoutservice.duration_mean               25                2  2024-01-15T23:23:00+00:00       0.017051  0.015366
1   checkoutservice       trace.from_checkoutservice.duration_p95               25                2  2024-01-15T23:20:00+00:00       0.078879  0.068836
2   checkoutservice      trace.from_frontendservice.duration_mean               25                2  2024-01-15T23:18:00+00:00       0.487127  0.468456
3   checkoutservice       trace.from_frontendservice.duration_p95               25                2  2024-01-15T23:12:00+00:00       1.087370  1.061202
4   checkoutservice                 trace.from_root.duration_mean               25                2  2024-01-15T23:07:00+00:00       0.079330  0.076188
5   checkoutservice                  trace.from_root.duration_p95               25                2  2024-01-15T23:07:00+00:00       0.192840  0.191463
6   checkoutservice        trace.to_checkoutservice.duration_mean               25                2  2024-01-15T23:23:00+00:00       0.017051  0.015366
7   checkoutservice         trace.to_checkoutservice.duration_p95               25                2  2024-01-15T23:20:00+00:00       0.078879  0.068836
8   checkoutservice        trace.to_currencyservice.duration_mean               25                2  2024-01-15T23:11:00+00:00       0.003715  0.003531
9   checkoutservice         trace.to_currencyservice.duration_p95               25                2  2024-01-15T23:21:00+00:00       0.000978  0.000421
10  checkoutservice           trace.to_emailservice.duration_mean               25                2  2024-01-15T23:20:00+00:00       0.000434  0.000337
11  checkoutservice            trace.to_emailservice.duration_p95               25                2  2024-01-15T23:20:00+00:00       0.001659  0.000786
12  checkoutservice         trace.to_paymentservice.duration_mean               25                2  2024-01-15T23:08:00+00:00       0.000499  0.000408
13  checkoutservice          trace.to_paymentservice.duration_p95               25                2  2024-01-15T23:12:00+00:00       0.001402  0.001087
14  checkoutservice  trace.to_productcatalogservice.duration_mean               25                2  2024-01-15T23:24:00+00:00       0.001316  0.000166
15  checkoutservice   trace.to_productcatalogservice.duration_p95               25                2  2024-01-15T23:15:00+00:00       0.000041  0.000036
16  currencyservice      trace.from_checkoutservice.duration_mean               25                2  2024-01-15T23:11:00+00:00       0.003715  0.003531
17  currencyservice       trace.from_checkoutservice.duration_p95               25                2  2024-01-15T23:21:00+00:00       0.000978  0.000421
18  currencyservice      trace.from_frontendservice.duration_mean               25                2  2024-01-15T23:07:00+00:00       0.001190  0.001140
19  currencyservice       trace.from_frontendservice.duration_p95               25                2  2024-01-15T23:17:00+00:00       0.000197  0.000194, 44, '2024-01-15T23:07:00+00:00')```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np

# Reuse df_trace if present; otherwise load and parse timestamps as UTC
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Candidate components and incident window (reuse if available)
try:
    candidates
except NameError:
    candidates = [
        'checkoutservice',
        'currencyservice',
        'emailservice',
        'productcatalogservice',
        'recommendationservice'
    ]
try:
    start, end
except NameError:
    start = pd.to_datetime('2024-01-15 23:03:19', utc=True)
    end   = pd.to_datetime('2024-01-15 23:33:19', utc=True)

# Restrict to candidate components
df_trace_cand = df_trace[df_trace['cmdb_id'].isin(candidates)].copy()

# Compute global p95 per (cmdb_id, trace_name) using full series (BEFORE window filtering)
trace_global_p95 = (
    df_trace_cand
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95))
)

# Filter trace points to the incident window and merge p95
df_trace_win = (
    df_trace_cand[(df_trace_cand['timestamp'] >= start) & (df_trace_cand['timestamp'] <= end)]
    .merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')
    .copy()
)

# Find points where value > p95 (strictly)
df_trace_win['above_p95'] = df_trace_win['value'] > df_trace_win['p95']

# For each pair that has any such points, get earliest timestamp where value > p95
df_above = df_trace_win[df_trace_win['above_p95']].copy()
earliest_per_pair = (
    df_above
    .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_above_ts'})
    .sort_values('earliest_above_ts', ascending=True)
)

# Build a dict of window groups sorted by timestamp for quick access
grouped_window = {
    key: g.sort_values('timestamp').reset_index(drop=True)
    for key, g in df_trace_win.groupby(['cmdb_id','trace_name'])
}

# For each pair with earliest_above_ts, collect up to two rows: the earliest-above row and the immediate next row if exists
rows = []
for _, row in earliest_per_pair.iterrows():
    cmdb = row['cmdb_id']
    tname = row['trace_name']
    earliest_ts = row['earliest_above_ts']
    key = (cmdb, tname)
    dfg = grouped_window.get(key)
    if dfg is None or dfg.empty:
        continue
    # find position(s) where timestamp equals earliest_ts
    mask = dfg['timestamp'] == earliest_ts
    if not mask.any():
        # fallback: find first row where value > p95
        pos_arr = np.where(dfg['value'] > dfg['p95'])[0]
        if pos_arr.size == 0:
            continue
        pos = int(pos_arr[0])
    else:
        pos = int(np.where(mask.values)[0][0])
    # append the earliest row
    rows.append({
        'timestamp': dfg.loc[pos, 'timestamp'],
        'cmdb_id': cmdb,
        'trace_name': tname,
        'value': dfg.loc[pos, 'value']
    })
    # append immediate next row if exists
    if pos + 1 < len(dfg):
        rows.append({
            'timestamp': dfg.loc[pos+1, 'timestamp'],
            'cmdb_id': cmdb,
            'trace_name': tname,
            'value': dfg.loc[pos+1, 'value']
        })

# Create result DataFrame, convert timestamps to ISO UTC strings
result_df = pd.DataFrame(rows)
if not result_df.empty:
    result_df['timestamp'] = result_df['timestamp'].dt.tz_convert('UTC').apply(lambda x: x.isoformat())

# Sort results by the group's earliest_above timestamp ascending (we already iterated in that order),
# but ensure rows are grouped accordingly and limit total rows to 50.
result_df_limited = result_df.head(50)

# Output compact table
result_df_limited

```
Out[8]:
```
```python
# Summarize the extracted earliest-above-p95 trace points (reuse result_df_limited from prior execution)
import pandas as pd

if 'result_df_limited' in globals() and not result_df_limited.empty:
    df = result_df_limited.copy()
    # Ensure timestamp parsed
    df['ts_dt'] = pd.to_datetime(df['timestamp'], utc=True)
    total_rows = len(df)
    earliest_ts = df['ts_dt'].min().isoformat()
    # Top involved services and trace examples
    top_services = df['cmdb_id'].drop_duplicates().tolist()[:6]
    top_traces = df['trace_name'].drop_duplicates().tolist()[:6]
    # Build summary
    summary = (
        f"Summary of trace points around the first above-p95 events (returned up to 50 rows):\n"
        f"- Total rows returned: {total_rows}\n"
        f"- Earliest above-p95 occurrence (among returned pairs): {earliest_ts} (UTC)\n"
        f"- Services present in these rows (examples): {', '.join(top_services)}\n"
        f"- Trace names present (examples): {', '.join(top_traces)}\n\n"
        f"- For each trace pair that exceeded its global p95, we returned up to two incident-window rows: the earliest sample that exceeded p95 and the immediate next timestamp (if any).\n"
        f"- Notable earliest entries include checkoutservice trace.from_root.duration_mean and duration_p95 at 2024-01-15T23:07:00+00:00, and several checkoutservice/currencyservice trace row_count and duration series clustered shortly after.\n\n"
        f"Suggested next step: inspect the returned rows around the earliest timestamps (e.g., 23:07 UTC and the ~23:12–23:24 UTC window) to see which spans/requests correspond to those slow samples and correlate them with metric/log anomalies."
    )
else:
    summary = "No results available to summarize (result_df_limited is missing or empty)."

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

timestamp          cmdb_id                            trace_name        value
0   2024-01-15T23:07:00+00:00  checkoutservice         trace.from_root.duration_mean     0.079330
1   2024-01-15T23:08:00+00:00  checkoutservice         trace.from_root.duration_mean     0.061966
2   2024-01-15T23:07:00+00:00  checkoutservice          trace.from_root.duration_p95     0.191713
3   2024-01-15T23:08:00+00:00  checkoutservice          trace.from_root.duration_p95     0.132603
4   2024-01-15T23:07:00+00:00  currencyservice  trace.from_frontendservice.row_count  2307.000000
..                        ...              ...                                   ...          ...
45  2024-01-15T23:15:00+00:00  checkoutservice  trace.from_checkoutservice.row_count   192.000000
46  2024-01-15T23:14:00+00:00  checkoutservice  trace.from_frontendservice.row_count    38.000000
47  2024-01-15T23:15:00+00:00  checkoutservice  trace.from_frontendservice.row_count    24.000000
48  2024-01-15T23:14:00+00:00  checkoutservice             trace.from_root.row_count   154.000000
49  2024-01-15T23:15:00+00:00  checkoutservice             trace.from_root.row_count    96.000000

[50 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import re
import numpy as np

# Reuse df_trace if already loaded; otherwise load and parse timestamps as UTC
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Candidate components and incident window (reuse if available)
try:
    candidates
except NameError:
    candidates = [
        'checkoutservice',
        'currencyservice',
        'emailservice',
        'productcatalogservice',
        'recommendationservice'
    ]
start = pd.to_datetime('2024-01-15 23:03:19', utc=True)
end   = pd.to_datetime('2024-01-15 23:33:19', utc=True)

# Restrict to candidate components and trace_name containing 'duration' (case-insensitive)
mask_comp = df_trace['cmdb_id'].isin(candidates)
mask_name = df_trace['trace_name'].str.contains('duration', case=False, na=False)
df_trace_sel = df_trace[mask_comp & mask_name].copy()

# Compute global p95 per (cmdb_id, trace_name) using the full series (BEFORE window filtering)
trace_p95 = (
    df_trace_sel
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95))
)

# Filter trace points to the incident window and merge p95 thresholds
df_win = df_trace_sel[(df_trace_sel['timestamp'] >= start) & (df_trace_sel['timestamp'] <= end)].copy()
df_win = df_win.merge(trace_p95, on=['cmdb_id','trace_name'], how='left')

# Select points strictly above p95
df_win['above_p95'] = df_win['value'] > df_win['p95']
df_above = df_win[df_win['above_p95']].copy()

# For each pair with any above-p95 points, get earliest timestamp and max in window and p95
agg = (
    df_above
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(earliest_above_ts=('timestamp','min'),
         max_in_window=('value','max'))
    .merge(trace_p95, on=['cmdb_id','trace_name'], how='left')
)

# Parse direction and peer from trace_name
def parse_direction_peer(tname):
    tn = str(tname).lower()
    direction = 'other'
    peer = ''
    # root detection
    if 'trace.from_root' in tn:
        direction = 'root'
    elif 'trace.to_' in tn:
        direction = 'to'
    elif 'trace.from_' in tn:
        direction = 'from'
    # peer extraction for to/from (but not root)
    m = re.search(r'trace\.(?:to|from)_([a-z0-9_]+)', tn)
    if m:
        peer = m.group(1)
        # if peer == 'root' we keep peer blank per instructions (not parseable)
        if peer == 'root':
            peer = ''
    return direction, peer

parsed = agg['trace_name'].apply(lambda x: parse_direction_peer(x))
agg[['direction','peer']] = pd.DataFrame(parsed.tolist(), index=agg.index)

# Format earliest timestamp to ISO UTC string
agg['earliest_above_p95'] = agg['earliest_above_ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat() if pd.notna(x) else np.nan)

# Select and order columns as requested
result_df = agg[['cmdb_id','trace_name','direction','peer','earliest_above_p95','max_in_window','p95']].copy()

# Sort by earliest_above_p95 ascending and limit to top 50 rows
result_df = result_df.sort_values(by='earliest_above_p95', ascending=True).head(50).reset_index(drop=True)

# Summary values
total_pairs_with_anomaly = int(agg.shape[0])
earliest_overall_ts = agg['earliest_above_ts'].min() if not agg['earliest_above_ts'].isna().all() else pd.NaT
earliest_overall_iso = earliest_overall_ts.isoformat() if pd.notna(earliest_overall_ts) else np.nan

# Return compact outputs
result_df, total_pairs_with_anomaly, earliest_overall_iso

```
Out[9]:
```
```python
# Build a concise plain-English summary string from previous trace-duration analysis results.
try:
    total = total_pairs_with_anomaly
except NameError:
    total = None
try:
    earliest = earliest_overall_iso
except NameError:
    earliest = None

summary = (
    f"Trace-duration anomaly summary (candidate services):\n"
    f"- Total (cmdb_id, trace_name) pairs with values > global p95 in the incident window: {total}\n"
    f"- Earliest above-p95 observed: {earliest} (UTC)\n\n"
    f"- Key observations:\n"
    f"  • The first above-p95 events appear at 2024-01-15T23:07:00Z, notably for checkoutservice (e.g., trace.from_root.duration_mean and duration_p95).\n"
    f"  • Many duration traces across services exceeded their p95s, including traces labeled 'from_frontendservice', 'from_root', and various 'to_<service>' paths for checkoutservice and currencyservice.\n"
    f"  • These above-p95 occurrences are distributed through the window (examples extend into ~23:24–23:28 UTC), indicating multiple short-lived slow-span events rather than a single sustained spike.\n\n"
    f"- Recommended next steps:\n"
    f"  1) Inspect the specific span samples around the earliest timestamps (e.g., 23:07 UTC) to identify which requests or downstream calls were slow.\n"
    f"  2) Correlate these trace spikes with metric anomalies (latency, CPU, workload) and log volume to narrow the root cause.\n"
    f"  3) Focus on inter-service paths involving checkoutservice and currencyservice, as many affected traces involve those interactions."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                     trace_name direction                   peer         earliest_above_p95  max_in_window       p95
0         checkoutservice                  trace.from_root.duration_mean      root                         2024-01-15T23:07:00+00:00       0.079330  0.076188
1         checkoutservice                   trace.from_root.duration_p95      root                         2024-01-15T23:07:00+00:00       0.192840  0.191463
2         currencyservice       trace.from_frontendservice.duration_mean      from        frontendservice  2024-01-15T23:07:00+00:00       0.001190  0.001140
3         checkoutservice          trace.to_paymentservice.duration_mean        to         paymentservice  2024-01-15T23:08:00+00:00       0.000499  0.000408
4         currencyservice       trace.from_checkoutservice.duration_mean      from        checkoutservice  2024-01-15T23:11:00+00:00       0.003715  0.003531
..                    ...                                            ...       ...                    ...                        ...            ...       ...
39  recommendationservice        trace.from_frontendservice.duration_p95      from        frontendservice  2024-01-15T23:26:00+00:00       0.086410  0.085503
40  productcatalogservice        trace.from_frontendservice.duration_p95      from        frontendservice  2024-01-15T23:26:00+00:00       0.000030  0.000029
41  recommendationservice  trace.from_recommendationservice.duration_p95      from  recommendationservice  2024-01-15T23:26:00+00:00       0.085908  0.085071
42  recommendationservice    trace.to_recommendationservice.duration_p95        to  recommendationservice  2024-01-15T23:26:00+00:00       0.085908  0.085071
43  productcatalogservice       trace.from_frontendservice.duration_mean      from        frontendservice  2024-01-15T23:28:00+00:00       0.000247  0.000231

[44 rows x 7 columns], 44, '2024-01-15T23:07:00+00:00')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np

# Reuse already-loaded data if available; otherwise load and parse timestamps as UTC
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Candidate services and numeric KPIs
candidates = [
    'checkoutservice',
    'currencyservice',
    'emailservice',
    'productcatalogservice',
    'recommendationservice'
]
kpis_numeric = ['cpu', 'latency-50', 'latency-90', 'mem', 'workload']

# Incident window (UTC)
start = pd.to_datetime('2024-01-15 23:03:19', utc=True)
end   = pd.to_datetime('2024-01-15 23:33:19', utc=True)

# ---------------------------
# Part A: trace-duration earliest above-p95 per service
# ---------------------------

# Select trace-duration rows for candidate services
df_trace_sel = df_trace[
    df_trace['cmdb_id'].isin(candidates) &
    df_trace['trace_name'].str.contains('duration', case=False, na=False)
].copy()

# Global p95 per (cmdb_id, trace_name) using full series (before window filtering)
trace_p95 = (
    df_trace_sel
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95))
)

# Filter trace points to the incident window and merge p95
df_trace_win = df_trace_sel[(df_trace_sel['timestamp'] >= start) & (df_trace_sel['timestamp'] <= end)].merge(
    trace_p95, on=['cmdb_id','trace_name'], how='left'
)

# Mark strictly above p95
df_trace_win['above_p95'] = df_trace_win['value'] > df_trace_win['p95']

# Earliest above-p95 per (cmdb_id, trace_name)
df_trace_above = df_trace_win[df_trace_win['above_p95']].copy()
if not df_trace_above.empty:
    earliest_per_pair = (
        df_trace_above
        .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp':'earliest_above_ts'})
    )
else:
    earliest_per_pair = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_above_ts'])

# Earliest above-p95 per cmdb_id (service)
if not earliest_per_pair.empty:
    earliest_per_service = (
        earliest_per_pair
        .groupby('cmdb_id', as_index=False)['earliest_above_ts']
        .min()
    )
else:
    # no above-p95 anywhere
    earliest_per_service = pd.DataFrame({'cmdb_id': candidates, 'earliest_above_ts': [pd.NaT]*len(candidates)})

# Ensure every candidate service is present
earliest_per_service = pd.DataFrame({'cmdb_id': candidates}).merge(
    earliest_per_service, on='cmdb_id', how='left'
)

# Convert earliest timestamps to ISO strings or NaN
earliest_per_service['earliest_trace_above_p95'] = earliest_per_service['earliest_above_ts'].dt.tz_convert('UTC').apply(
    lambda x: x.isoformat() if pd.notna(x) else np.nan
)

# ---------------------------
# Part B: numeric KPI severity per service
# ---------------------------

# Select numeric KPI rows for candidate services
df_metric_sel = df_metric[
    df_metric['cmdb_id'].isin(candidates) &
    df_metric['kpi_name'].isin(kpis_numeric)
].copy()

# Compute global p95 and p5 per (cmdb_id, kpi_name) using full series (before window filtering)
thresholds = (
    df_metric_sel
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
)

# Filter metric points to the incident window
df_metric_win = df_metric_sel[(df_metric_sel['timestamp'] >= start) & (df_metric_sel['timestamp'] <= end)].copy()

# Compute min/max in window per pair
extremes = (
    df_metric_win
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(max_in_window=('value','max'), min_in_window=('value','min'))
)

# Merge thresholds with window extremes (keep all thresholded pairs)
pairs = thresholds.merge(extremes, on=['cmdb_id','kpi_name'], how='left')

# Compute severity per pair
def compute_severity(row):
    p95 = row['p95']
    p5  = row['p5']
    maxw = row['max_in_window']
    minw = row['min_in_window']
    above_sev = 0.0
    below_sev = 0.0
    # above breach
    if pd.notna(maxw) and pd.notna(p95) and p95 > 0 and maxw > p95:
        above_sev = (maxw - p95) / p95
    # below breach
    if pd.notna(minw) and pd.notna(p5) and p5 > 0 and minw < p5:
        below_sev = (p5 - minw) / p5
    # choose max
    if above_sev >= below_sev and above_sev > 0:
        return pd.Series({'severity': float(above_sev), 'side_used': 'above'})
    elif below_sev > 0:
        return pd.Series({'severity': float(below_sev), 'side_used': 'below'})
    else:
        return pd.Series({'severity': 0.0, 'side_used': np.nan})

sev_df = pairs.join(pairs.apply(compute_severity, axis=1))

# For each service, pick the KPI with maximum severity
# Ensure all candidate services appear even if they had no KPIs
sev_df_all = pd.DataFrame({'cmdb_id': candidates}).merge(sev_df, on='cmdb_id', how='left')

# Fill missing severity with 0
sev_df_all['severity'] = sev_df_all['severity'].fillna(0.0)

# For selecting top KPI per service, group and pick row with max severity
top_kpis = (
    sev_df_all
    .sort_values(['cmdb_id','severity','kpi_name'], ascending=[True, False, True])
    .groupby('cmdb_id', as_index=False)
    .first()[['cmdb_id','kpi_name','severity','side_used']]
    .rename(columns={'kpi_name':'top_numeric_kpi', 'severity':'top_severity'})
)

# For services where top_severity == 0, set fields to NaN as requested
top_kpis.loc[top_kpis['top_severity'] <= 0, ['top_numeric_kpi','top_severity','side_used']] = np.nan

# ---------------------------
# Combine Part A and Part B into final compact table
# ---------------------------

final_df = earliest_per_service[['cmdb_id','earliest_trace_above_p95']].merge(
    top_kpis, on='cmdb_id', how='left'
)[['cmdb_id','earliest_trace_above_p95','top_numeric_kpi','top_severity','side_used']]

# Compute service(s) with overall minimum earliest_trace_above_p95 (exclude NaN)
valid_earliest = earliest_per_service.dropna(subset=['earliest_above_ts']).copy()
if not valid_earliest.empty:
    overall_min_ts = valid_earliest['earliest_above_ts'].min()
    overall_min_iso = overall_min_ts.isoformat()
    services_first = valid_earliest[valid_earliest['earliest_above_ts'] == overall_min_ts]['cmdb_id'].tolist()
else:
    overall_min_iso = np.nan
    services_first = []

# Keep output compact: one row per candidate service (limit 5)
final_df, services_first, overall_min_iso

```
Out[10]:
```
```python
# Build a plain-English summary from the computed results (reuse variables: final_df, services_first, overall_min_iso)
summary_lines = []
summary_lines.append("Service-level summary (candidate services):")
for _, r in final_df.iterrows():
    svc = r['cmdb_id']
    t = r['earliest_trace_above_p95'] if pd.notna(r['earliest_trace_above_p95']) else "NaN"
    kpi = r['top_numeric_kpi'] if pd.notna(r['top_numeric_kpi']) else "NaN"
    sev = f"{r['top_severity']:.6g}" if pd.notna(r['top_severity']) else "NaN"
    side = r['side_used'] if pd.notna(r['side_used']) else "NaN"
    summary_lines.append(f"- {svc}: earliest_trace_above_p95={t}, top_numeric_kpi={kpi}, top_severity={sev}, side={side}")

if services_first:
    first_services = ", ".join(services_first)
else:
    first_services = "None"

overall_ts = overall_min_iso if pd.notna(overall_min_iso) else "NaN"

summary_lines.append("")
summary_lines.append(f"Service(s) that first showed trace-duration spikes: {first_services}")
summary_lines.append(f"Earliest trace-duration above-p95 timestamp overall: {overall_ts} (UTC)")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id   earliest_trace_above_p95 top_numeric_kpi  top_severity side_used
0        checkoutservice  2024-01-15T23:07:00+00:00      latency-50      0.150377     below
1        currencyservice  2024-01-15T23:07:00+00:00      latency-90      0.109573     above
2           emailservice  2024-01-15T23:11:00+00:00        workload      0.191609     below
3  productcatalogservice  2024-01-15T23:15:00+00:00             cpu      0.043822     below
4  recommendationservice  2024-01-15T23:11:00+00:00      latency-90      0.080353     above, ['checkoutservice', 'currencyservice'], '2024-01-15T23:07:00+00:00')```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np

# Reuse df_trace if available; otherwise load and parse timestamps as UTC
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Define narrow window
start = pd.to_datetime('2024-01-15 23:06:00', utc=True)
end   = pd.to_datetime('2024-01-15 23:12:00', utc=True)

# Filter to window
df_win = df_trace[(df_trace['timestamp'] >= start) & (df_trace['timestamp'] <= end)].copy()

# Build masks for the requested conditions (case-insensitive)
mask_checkout = (
    (df_win['cmdb_id'] == 'checkoutservice') &
    df_win['trace_name'].str.contains(
        r'to_currencyservice|from_root|from_frontendservice|to_paymentservice',
        case=False, regex=True, na=False
    )
)

mask_currency = (
    (df_win['cmdb_id'] == 'currencyservice') &
    df_win['trace_name'].str.contains(
        r'from_checkoutservice|from_frontendservice|from_root|to_checkoutservice',
        case=False, regex=True, na=False
    )
)

mask = mask_checkout | mask_currency

# Select requested columns, convert timestamp to ISO UTC, sort, and limit to 50 rows
result = (
    df_win.loc[mask, ['timestamp', 'cmdb_id', 'trace_name', 'value']]
    .sort_values('timestamp', ascending=True)
    .reset_index(drop=True)
)

if not result.empty:
    result['timestamp'] = result['timestamp'].dt.tz_convert('UTC').apply(lambda x: x.isoformat())

result = result.head(50)

result

```
Out[11]:
```
Summary of trace activity in 2024-01-15 23:06:00 — 23:12:00 UTC (filtered traces for checkoutservice and currencyservice):

- Earliest matching traces appear at 2024-01-15T23:06:00+00:00.
- Both checkoutservice and currencyservice show trace activity in this window:
  - checkoutservice: multiple duration traces from frontend and from_root around 23:06–23:07 (examples: trace.from_frontendservice.duration_mean = 0.254664 at 23:06:00; trace.from_root.duration_mean = 0.079330 and trace.from_root.duration_p95 = 0.191713 at 23:07:00; another frontend duration_mean = 0.372423 at 23:07:00).
  - currencyservice: several from_root and from_frontendservice traces at 23:06:00 with small duration values and row_count entries (example: trace.from_root.row_count = 8 at 23:06:00).
- Result set returned up to 50 rows; timestamps are clustered around 23:06–23:07 with repeated samples for the same trace names.

Interpretation / next steps:
- Checkoutservice shows short-lived duration spikes in the very early part of the window (≈23:06–23:07 UTC) and should be investigated first.
- Correlate these trace samples with the metric anomalies (latency/CPU/workload) and logs at the same timestamps to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp          cmdb_id                                trace_name      value
0   2024-01-15T23:06:00+00:00  checkoutservice  trace.from_frontendservice.duration_mean   0.254664
1   2024-01-15T23:06:00+00:00  currencyservice                 trace.from_root.row_count   8.000000
2   2024-01-15T23:06:00+00:00  currencyservice                trace.from_root.error_rate   0.000000
3   2024-01-15T23:06:00+00:00  currencyservice              trace.from_root.duration_p95   0.000070
4   2024-01-15T23:06:00+00:00  currencyservice             trace.from_root.duration_mean   0.000048
..                        ...              ...                                       ...        ...
45  2024-01-15T23:07:00+00:00  checkoutservice      trace.from_frontendservice.row_count  28.000000
46  2024-01-15T23:07:00+00:00  checkoutservice             trace.from_root.duration_mean   0.079330
47  2024-01-15T23:07:00+00:00  checkoutservice              trace.from_root.duration_p95   0.191713
48  2024-01-15T23:07:00+00:00  checkoutservice                trace.from_root.error_rate   0.000000
49  2024-01-15T23:07:00+00:00  checkoutservice  trace.from_frontendservice.duration_mean   0.372423

[50 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```